# 7B 线步骤①冒烟 v5（CPU 验证轮 / GPU 实测轮共用）

v4 定因三连：cu128 index 最高只有 torch 2.11（改 PyPI 装，linux 自带 CUDA 依赖）；requirements 在 clone 前被引用（顺序修正）；torch 2.11+cpu 未替换（1 的连带）。
CPU 轮：clone → PyPI torch → requirements → 导入链 + tokenizer（只下 tokenizer 文件，省 5GB）；GPU 轮重推后自动跑全量 + 冒烟。

In [ ]:
# Cell 1: 浅克隆（先行——requirements 在仓库里）
import os
os.environ['GIT_TERMINAL_PROMPT'] = '0'
import subprocess
r = subprocess.run('git clone --depth 1 https://github.com/Huzhenghua/emotion_model.git',
                   shell=True, capture_output=True, text=True)
print('clone exit:', r.returncode)
print(r.stdout[-500:], r.stderr[-500:])
assert r.returncode == 0, 'clone 失败'
%cd emotion_model
!git log --oneline -1

In [ ]:
# Cell 2: torch 从 PyPI 装（>=2.13 过 gptqmodel 安全检查；linux wheel 自带 CUDA nvidia 依赖）
r = subprocess.run('pip install -q "torch>=2.13,<2.16"', shell=True, capture_output=True, text=True)
print('torch install exit:', r.returncode)
print(r.stderr[-2000:])
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
# Cell 3: 其余依赖 + 导入链验证（gptqmodel cp313 wheel 是否存在在此见分晓）
r = subprocess.run('pip install -q -r requirements.txt', shell=True, capture_output=True, text=True)
print('requirements install exit:', r.returncode)
print(r.stderr[-2000:])
import transformers, gptqmodel, torch as t2
print('transformers', transformers.__version__, '| gptqmodel', gptqmodel.__version__)
print('torch 仍未被替换:', t2.__version__, t2.cuda.is_available())
assert t2.__version__ >= '2.13', 'torch 被降级替换！'

In [ ]:
# Cell 4: 模型文件——CPU 轮只下 tokenizer（~10MB），GPU 轮下全量（~5GB）
import torch, subprocess
if torch.cuda.is_available():
    cmd = 'modelscope download --model Qwen/Qwen2.5-7B-Instruct-AWQ --local_dir Qwen2.5-7B-Instruct-AWQ'
else:
    cmd = ('modelscope download --model Qwen/Qwen2.5-7B-Instruct-AWQ '
           'tokenizer.json tokenizer_config.json vocab.json merges.txt generation_config.json '
           '--local_dir Qwen2.5-7B-Instruct-AWQ')
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print('download exit:', r.returncode)
print(r.stderr[-1000:])
!ls Qwen2.5-7B-Instruct-AWQ/

In [ ]:
# Cell 5: tokenizer 验证（CPU 安全）
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen2.5-7B-Instruct-AWQ')
print('tokenizer OK, vocab', len(tok))

In [ ]:
# Cell 6: 冒烟（仅 GPU 轮）+ 环境报告落盘
import torch, subprocess, json
report = {'torch': torch.__version__, 'cuda': torch.cuda.is_available()}
if torch.cuda.is_available():
    r = subprocess.run('python eval/smoke_7b_load.py --model Qwen2.5-7B-Instruct-AWQ --max-new-tokens 128',
                       shell=True, capture_output=True, text=True)
    report['smoke_exit'] = r.returncode
    open('/kaggle/working/smoke_7b.log', 'w').write(r.stdout + '\n===STDERR===\n' + r.stderr[-5000:])
    print(r.stdout)
else:
    report['smoke'] = 'skipped (CPU round)'
json.dump(report, open('/kaggle/working/env_report.json', 'w'), indent=2)
print(json.dumps(report, indent=2))